In [1]:
# -*- coding: utf-8 -*-
"""
PIPELINE HUẤN LUYỆN HOÀN CHỈNH (v2 - Augmentation Nâng cao)
Bài toán: Phát hiện "Đường lưỡi bò" (Nine Dash Line)
Kiến trúc: Faster R-CNN (ResNet-50) kết hợp Chắt lọc Tri thức.
Cải tiến: Sử dụng thư viện Albumentations để tăng cường dữ liệu nâng cao.
"""

# =====================================================================================
# PHẦN 0: IMPORT CÁC THƯ VIỆN CẦN THIẾT
# =====================================================================================
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
from torchvision.models.detection import FasterRCNN, fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import box_iou, MultiScaleRoIAlign
from torch.utils.data import DataLoader
from torchvision.datasets import CocoDetection
from PIL import Image
import os
import numpy as np
# THAY ĐỔI: Import thư viện Albumentations
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchmetrics.detection import MeanAveragePrecision
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# =====================================================================================
# PHẦN 1: ĐỊNH NGHĨA LỚP DATASET VÀ AUGMENTATION
# =====================================================================================

# THAY ĐỔI: Định nghĩa pipeline augmentation bằng Albumentations
def get_transform(train):
    """
    Định nghĩa các phép biến đổi và augmentation cho ảnh bằng Albumentations.
    """
    if train:
        # SỬA LỖI: Sử dụng định dạng 'pascal_voc' để nhất quán với mô hình.
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.Affine(scale=(0.9, 1.1), translate_percent=(-0.0625, 0.0625), rotate=(-15, 15), p=0.5),
            A.RandomBrightnessContrast(p=0.2),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'], min_visibility=0.25))
    else:
        return A.Compose([
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

# THAY ĐỔI: Cập nhật lớp Dataset để tương thích với Albumentations
class NineDashLineDataset(CocoDetection):
    """
    Lớp Dataset tùy chỉnh để tải bộ dữ liệu TTCSS v2 và áp dụng Albumentations.
    """
    def __init__(self, root, annFile, transforms=None):
        super(NineDashLineDataset, self).__init__(root, annFile)
        self._transforms = transforms

    def __getitem__(self, idx):
        # Lấy ảnh và thông tin annotation gốc.
        img, target = super(NineDashLineDataset, self).__getitem__(idx)
        
        # Chuyển đổi bounding box và labels sang định dạng mà Albumentations yêu cầu.
        boxes = []
        labels = []
        w, h = img.size
        for obj in target:
            # SỬA LỖI: Chuyển đổi sang định dạng 'pascal_voc': [x_min, y_min, x_max, y_max]
            x, y, box_w, box_h = obj['bbox']
            x_max = x + box_w
            y_max = y + box_h
            
            # SỬA LỖI: Clip tọa độ để đảm bảo chúng nằm trong ranh giới ảnh TRƯỚC KHI augmentation.
            x = max(0, x)
            y = max(0, y)
            x_max = min(w, x_max)
            y_max = min(h, y_max)
            
            # Chỉ thêm box nếu nó vẫn có kích thước hợp lệ sau khi clip
            if x_max > x and y_max > y:
                boxes.append([x, y, x_max, y_max])
                labels.append(obj['category_id'])

        # Tạo dictionary để truyền vào pipeline augmentation.
        transformed = self._transforms(image=np.array(img), bboxes=boxes, labels=labels)
        
        image = transformed['image']
        transformed_boxes = transformed['bboxes']
        transformed_labels = transformed['labels']

        # Xử lý trường hợp augmentation loại bỏ tất cả các bounding box.
        if len(transformed_boxes) > 0:
            boxes_tensor = torch.as_tensor(transformed_boxes, dtype=torch.float32)
        else:
            # Nếu không còn box nào, tạo một tensor rỗng với shape (0, 4).
            boxes_tensor = torch.zeros((0, 4), dtype=torch.float32)

        # Tạo dictionary target cuối cùng.
        final_target = {
            "boxes": boxes_tensor,
            "labels": torch.as_tensor(transformed_labels, dtype=torch.int64),
            "image_id": torch.tensor([self.ids[idx]])
        }
            
        return image, final_target

# =====================================================================================
# PHẦN 2: ĐỊNH NGHĨA MÔ HÌNH NÂNG CAO
# =====================================================================================
class EnhancedFasterRCNN(FasterRCNN):
    """
    Lớp mô hình kế thừa từ FasterRCNN của torchvision.
    Mục đích: Tùy chỉnh phương thức forward để trả về các thông tin cần thiết
    cho việc tính toán Knowledge Distillation Loss.
    """
    def forward(self, images, targets=None):
        if self.training and targets is None:
            raise ValueError("In training mode, targets should be passed")

        original_image_sizes = [img.shape[-2:] for img in images]
        images, targets = self.transform(images, targets)

        features = self.backbone(images.tensors)

        proposals, proposal_losses = self.rpn(images, features, targets)

        detections, detector_losses = self.roi_heads(features, proposals, images.image_sizes, targets)

        detections = self.transform.postprocess(detections, images.image_sizes, original_image_sizes)

        losses = {}
        losses.update(detector_losses)
        losses.update(proposal_losses)

        if self.training:
            return losses, features, proposals, detections, images.image_sizes
        
        return detections

def get_model(num_classes):
    """
    Hàm khởi tạo mô hình EnhancedFasterRCNN với backbone ResNet-50-FPN.
    """
    model = fasterrcnn_resnet50_fpn(weights='DEFAULT')
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    model.__class__ = EnhancedFasterRCNN
    return model

# =====================================================================================
# PHẦN 3: ĐỊNH NGHĨA HÀM MẤT MÁT CHẮT LỌC TRI THỨC
# =====================================================================================
class KnowledgeDistillationLoss(nn.Module):
    """
    Lớp tính toán KD Loss dựa trên việc so sánh đặc trưng của các VÙNG ĐỀ XUẤT (proposals)
    với đặc trưng của các VÙNG ĐÚNG (ground-truth), được trích xuất từ backbone features.
    """
    def __init__(self, beta1=1.0, beta2=0.1, iou_threshold=0.3):
        super().__init__()
        self.beta1 = beta1
        self.beta2 = beta2
        self.iou_threshold = iou_threshold
        self.roi_align = MultiScaleRoIAlign(featmap_names=['0', '1', '2', '3'], 
                                            output_size=7, 
                                            sampling_ratio=2)

    def _match_proposals_to_ground_truth(self, proposals, gt_boxes, iou_threshold):
        if proposals.shape[0] == 0 or gt_boxes.shape[0] == 0:
            return [], []
        iou_matrix = box_iou(proposals, gt_boxes)
        max_iou_per_gt, max_iou_indices_per_gt = iou_matrix.max(dim=0)
        matched_proposal_indices = []
        matched_gt_indices = []
        for i, iou in enumerate(max_iou_per_gt):
            if iou >= iou_threshold:
                proposal_idx = max_iou_indices_per_gt[i].item()
                if proposal_idx not in matched_proposal_indices:
                    matched_proposal_indices.append(proposal_idx)
                    matched_gt_indices.append(i)
        return matched_proposal_indices, matched_gt_indices

    def forward(self, features, proposals, targets, image_sizes):
        all_gt_embeddings = []
        all_pred_embeddings = []

        for i, (target, proposal) in enumerate(zip(targets, proposals)):
            gt_boxes = target['boxes']
            matched_proposal_indices, matched_gt_indices = self._match_proposals_to_ground_truth(
                proposal, gt_boxes, self.iou_threshold
            )
            if not matched_proposal_indices:
                continue

            matched_proposals = proposal[matched_proposal_indices]
            matched_gt_boxes = gt_boxes[matched_gt_indices]

            pred_aligned_features = self.roi_align(features, [matched_proposals], [image_sizes[i]])
            gt_aligned_features = self.roi_align(features, [matched_gt_boxes], [image_sizes[i]])

            pred_embeddings = F.adaptive_avg_pool2d(pred_aligned_features, (1, 1)).flatten(start_dim=1)
            gt_embeddings = F.adaptive_avg_pool2d(gt_aligned_features, (1, 1)).flatten(start_dim=1)
            
            pred_embeddings = F.normalize(pred_embeddings, p=2, dim=1)
            gt_embeddings = F.normalize(gt_embeddings, p=2, dim=1)
            
            all_pred_embeddings.append(pred_embeddings)
            all_gt_embeddings.append(gt_embeddings)

        if not all_gt_embeddings:
            return torch.tensor(0.0, device=proposals[0].device)

        final_gt_embeddings = torch.cat(all_gt_embeddings, dim=0)
        final_pred_embeddings = torch.cat(all_pred_embeddings, dim=0)

        loss_l1 = F.l1_loss(final_pred_embeddings, final_gt_embeddings)
        SR = torch.matmul(final_pred_embeddings, final_pred_embeddings.T)
        SI = torch.matmul(final_gt_embeddings, final_gt_embeddings.T)
        loss_irm = torch.norm(SR - SI, p='fro')

        return self.beta1 * loss_l1 + self.beta2 * loss_irm

# =====================================================================================
# PHẦN 4: UTILITIES VÀ VÒNG LẶP HUẤN LUYỆN
# =====================================================================================
def collate_fn(batch):
    return tuple(zip(*batch))

def train_one_epoch(model, optimizer, data_loader, device, epoch, kd_loss_fn):
    model.train()
    kd_loss_fn.to(device)
    total_main_loss = 0
    total_kd_loss = 0
    
    loop = tqdm(data_loader, desc=f"Epoch {epoch+1} Training")
    for i, (images, targets) in enumerate(loop):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict, features, proposals, detections, image_sizes = model(images, targets)
        
        main_losses = sum(loss for loss in loss_dict.values())
        
        kd_loss = kd_loss_fn(features, proposals, targets, image_sizes)
        
        total_loss = main_losses + kd_loss
        
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        
        loop.set_postfix(main_loss=main_losses.item(), kd_loss=kd_loss.item(), total_loss=total_loss.item())
                  
        total_main_loss += main_losses.item()
        if isinstance(kd_loss, torch.Tensor):
            total_kd_loss += kd_loss.item()
        
    avg_main_loss = total_main_loss / len(data_loader)
    avg_kd_loss = total_kd_loss / len(data_loader)
    print(f"--- Kết thúc Epoch {epoch+1} Training ---")
    print(f"  Loss chính trung bình: {avg_main_loss:.4f}")
    print(f"  KD Loss trung bình: {avg_kd_loss:.4f}")
    print(f"  Tổng Loss trung bình: {avg_main_loss + avg_kd_loss:.4f}")

@torch.no_grad()
def evaluate(model, data_loader, device):
    model.eval()
    metric = MeanAveragePrecision(box_format='xyxy')
    metric.to(device)

    print("\n--- Bắt đầu Đánh giá (Validation) ---")
    for images, targets in tqdm(data_loader, desc="Evaluating"):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        predictions = model(images)
        metric.update(predictions, targets)

    results = metric.compute()
    print(f"--- Kết quả Đánh giá ---")
    for k, v in results.items():
        if isinstance(v, torch.Tensor):
            print(f"  {k}: {v.item():.4f}")
    ar_100 = results["mar_100"].item()
    print(f"Average Recall @100 (AR@100): {ar_100:.4f}")
    return results

# =====================================================================================
# PHẦN 5: HÀM MAIN ĐỂ CHẠY TOÀN BỘ PIPELINE
# =====================================================================================
def main():
    # --- 1. Thiết lập các tham số ---
    TRAIN_IMG_DIR = '/kaggle/input/fasterrcnn-dataset/dataset/train/images'
    TRAIN_ANN_FILE = '/kaggle/input/fasterrcnn-dataset/dataset/train/_annotations.coco.json'
    VALID_IMG_DIR = '/kaggle/input/fasterrcnn-dataset/dataset/valid/images'
    VALID_ANN_FILE = '/kaggle/input/fasterrcnn-dataset/dataset/valid/_annotations.coco.json'
    
    if not os.path.exists(TRAIN_IMG_DIR) or not os.path.exists(VALID_IMG_DIR):
        print("="*50)
        print("LỖI: Không tìm thấy thư mục dữ liệu.")
        print(f"Vui lòng kiểm tra lại các đường dẫn: '{TRAIN_IMG_DIR}' và '{VALID_IMG_DIR}'")
        print("Hãy chắc chắn bạn đã cung cấp đúng đường dẫn đến bộ dữ liệu trên Kaggle.")
        print("="*50)
        return

    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    num_classes = 2 
    num_epochs = 20 # Tăng số epochs vì augmentation yêu cầu học lâu hơn
    batch_size = 4 # Có thể tăng batch size nếu GPU cho phép
    learning_rate = 0.005

    # --- 2. Chuẩn bị Dữ liệu ---
    print("Đang chuẩn bị dữ liệu...")
    dataset_train = NineDashLineDataset(root=TRAIN_IMG_DIR,
                                        annFile=TRAIN_ANN_FILE,
                                        transforms=get_transform(train=True))
    dataset_valid = NineDashLineDataset(root=VALID_IMG_DIR,
                                        annFile=VALID_ANN_FILE,
                                        transforms=get_transform(train=False))
    
    data_loader_train = DataLoader(dataset_train,
                                   batch_size=batch_size,
                                   shuffle=True,
                                   num_workers=2,
                                   collate_fn=collate_fn)
    data_loader_valid = DataLoader(dataset_valid,
                                   batch_size=1,
                                   shuffle=False,
                                   num_workers=2,
                                   collate_fn=collate_fn)
    
    print(f"Đã tải {len(dataset_train)} ảnh huấn luyện và {len(dataset_valid)} ảnh validation.")

    # --- 3. Khởi tạo Mô hình và Trình tối ưu hóa ---
    print("Đang khởi tạo mô hình...")
    model = get_model(num_classes)
    model.to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.SGD(params, lr=learning_rate, momentum=0.9, weight_decay=0.0005)
    
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer,
                                                   step_size=5,
                                                   gamma=0.1)

    # --- 4. Khởi tạo Hàm mất mát Chắt lọc Tri thức ---
    kd_loss_fn = KnowledgeDistillationLoss(beta1=1, beta2=0.5, iou_threshold=0.5)
    kd_loss_fn.to(device)
    
    # --- 5. Bắt đầu Vòng lặp Huấn luyện và Đánh giá ---
    print("\n" + "="*20 + " BẮT ĐẦU HUẤN LUYỆN VÀ ĐÁNH GIÁ " + "="*20)
    for epoch in range(num_epochs):
        train_one_epoch(model, optimizer, data_loader_train, device, epoch, kd_loss_fn)
        
        lr_scheduler.step()
        
        evaluate(model, data_loader_valid, device)
        
        torch.save(model.state_dict(), f'model_epoch_{epoch+1}.pth')
        print(f"Đã lưu checkpoint: model_epoch_{epoch+1}.pth")

    print("\n" + "="*20 + " HOÀN THÀNH HUẤN LUYỆN " + "="*20)

if __name__ == '__main__':
    main()

PyTorch version: 2.6.0+cu124
Torchvision version: 0.21.0+cu124
CUDA available: True
Đang chuẩn bị dữ liệu...
loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!
Đã tải 752 ảnh huấn luyện và 84 ảnh validation.
Đang khởi tạo mô hình...


/usr/local/lib/python3.11/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100%|██████████| 160M/160M [00:00<00:00, 208MB/s]



==================== BẮT ĐẦU HUẤN LUYỆN VÀ ĐÁNH GIÁ ====================


Epoch 1 Training: 100%|██████████| 188/188 [02:22<00:00,  1.32it/s, kd_loss=0.00731, main_loss=0.174, total_loss=0.182]


--- Kết thúc Epoch 1 Training ---
  Loss chính trung bình: 0.2191
  KD Loss trung bình: 0.0308
  Tổng Loss trung bình: 0.2499

--- Bắt đầu Đánh giá (Validation) ---


Evaluating: 100%|██████████| 84/84 [00:06<00:00, 13.18it/s]


... (73 outputs left out)

Epoch 20 Training: 100%|██████████| 188/188 [02:21<00:00,  1.33it/s, kd_loss=0.00124, main_loss=0.0717, total_loss=0.0729]


--- Kết thúc Epoch 20 Training ---
  Loss chính trung bình: 0.0903
  KD Loss trung bình: 0.0045
  Tổng Loss trung bình: 0.0948

--- Bắt đầu Đánh giá (Validation) ---


Evaluating: 100%|██████████| 84/84 [00:06<00:00, 13.85it/s]


--- Kết quả Đánh giá ---
  map: 0.5082
  map_50: 0.9392
  map_75: 0.4469
  map_small: 0.4000
  map_medium: 0.4763
  map_large: 0.5180
  mar_1: 0.5488
  mar_10: 0.5726
  mar_100: 0.5726
  mar_small: 0.4000
  mar_medium: 0.5100
  mar_large: 0.5836
  map_per_class: -1.0000
  mar_100_per_class: -1.0000
  classes: 1.0000
Average Recall @100 (AR@100): 0.5726
Đã lưu checkpoint: model_epoch_20.pth

==================== HOÀN THÀNH HUẤN LUYỆN ====================
